# Solutions · 14 · Systems of nonlinear equations

Worked solutions to the exercises of [notebook 14](../notebooks/14_nonlinear_systems.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmath is missing): install it from GitHub
    import engmath
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmath"], check=True)
import numpy as np
import matplotlib.pyplot as plt
import engmath
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")
from engmath import nonlinear, roots
z = np.array([120.0, 100.0, 60.0])
Lp = np.array([1000.0, 800.0, 1200.0]); Dp = np.array([0.30, 0.25, 0.20])
g = 9.81
k_const = 8 * 0.02 * Lp / (g * np.pi**2 * Dp**5)
def F_const(v):
    H, Q = v[0], v[1:]
    return np.r_[z - H - k_const * Q * np.abs(Q), Q.sum()]
sol_const = nonlinear.newton_system(F_const, [90.0, 0.1, 0.0, -0.1])

## Exercise 1

Replace the constant friction factor by the Colebrook equation (notebook 02) for each pipe
   (commercial steel, $\varepsilon$ = 0.045 mm; water $\nu$ = 1.0 × 10⁻⁶ m²/s), which puts a root solve
   inside every evaluation of $F$. How much do the flows change?

In [2]:
eps, nu = 0.045e-3, 1.0e-6
def friction(Q, D):
    Re = max(4*abs(Q)/(np.pi*D*nu), 4000.0)            # guard: Colebrook needs turbulent flow
    return roots.brent(lambda f: 1/np.sqrt(f) + 2*np.log10(eps/D/3.7 + 2.51/(Re*np.sqrt(f))), 1e-4, 0.1).root
def F_cole(v):
    H, Q = v[0], v[1:]
    f = np.array([friction(q, d) for q, d in zip(Q, Dp)])
    k = 8 * f * Lp / (g * np.pi**2 * Dp**5)
    return np.r_[z - H - k * Q * np.abs(Q), Q.sum()]
sol = nonlinear.newton_system(F_cole, sol_const.x)
print(f"junction head: constant f {sol_const.x[0]:.2f} m, Colebrook {sol.x[0]:.2f} m")
for i in range(3):
    print(f"pipe {i+1}: {sol_const.x[i+1]*1000:7.1f} L/s -> {sol.x[i+1]*1000:7.1f} L/s, "
          f"friction factor {friction(sol.x[i+1], Dp[i]):.4f}")

junction head: constant f 105.21 m, Colebrook 105.66 m
pipe 1:   147.5 L/s ->   170.8 L/s, friction factor 0.0145
pipe 2:   -62.0 L/s ->   -73.0 L/s, friction factor 0.0157
pipe 3:   -85.4 L/s ->   -97.8 L/s, friction factor 0.0154


With Colebrook the friction factors fall below the assumed 0.02 for these smooth steel pipes at high
Reynolds numbers, so all flows increase. The solution method is unchanged: $F$ simply contains inner root
solves. Starting from the constant-$f$ solution (a *continuation* strategy) gives Newton a good start.

## Exercise 2

Add a fourth reservoir at 90 m, connected by a 900 m, 0.25 m pipe. Which reservoirs now supply water?

In [3]:
z4 = np.r_[z, 90.0]; L4 = np.r_[Lp, 900.0]; D4 = np.r_[Dp, 0.25]
k4 = 8 * 0.02 * L4 / (g * np.pi**2 * D4**5)
F4 = lambda v: np.r_[z4 - v[0] - k4 * v[1:] * np.abs(v[1:]), v[1:].sum()]
sol4 = nonlinear.newton_system(F4, [90.0, 0.1, 0.0, -0.1, 0.0])
print(f"junction head {sol4.x[0]:.2f} m (converged: {sol4.converged})")
for i, q in enumerate(sol4.x[1:], start=1):
    print(f"reservoir {i} ({z4[i-1]:.0f} m): {'supplies' if q > 0 else 'receives'} {abs(q)*1000:.1f} L/s")

junction head 100.11 m (converged: True)
reservoir 1 (120 m): supplies 171.0 L/s
reservoir 2 (100 m): receives 9.1 L/s
reservoir 3 (60 m): receives 80.5 L/s
reservoir 4 (90 m): receives 81.5 L/s


The rule is simple once $H$ is known: reservoirs **above** the junction head supply water, those below
receive it. Adding the 90 m reservoir lowers the junction head (from 105.2 m to 100.1 m), because it
offers another outlet. The flow into the 100 m reservoir drops to a trickle: the junction head is now
barely above it, and a slightly larger fourth pipe would reverse that flow.

## Exercise 3

Start from $H = 150$ m (above every reservoir). Does Newton still find the solution, with and without
   the line search? Look at the residual history to explain.

In [4]:
for ls in (False, True):
    r = nonlinear.newton_system(F_const, [150.0, 0.1, 0.0, -0.1], line_search=ls, maxiter=50)
    print(f"line search {ls!s:5}: converged {r.converged} in {r.iterations} iterations, H = {r.x[0]:.2f} m; "
          f"residuals {np.array2string(np.array(r.residual_history[:6]), precision=1)}")

line search False: converged True in 6 iterations, H = 105.21 m; residuals [6.8e+01 1.9e+01 2.7e+00 1.3e-01 3.2e-04 2.0e-09]
line search True : converged True in 6 iterations, H = 105.21 m; residuals [6.8e+01 1.9e+01 2.7e+00 1.3e-01 3.2e-04 2.0e-09]


Both runs are identical: from 150 m every full Newton step already reduces $\|F\|$ enough, so the line
search never intervenes, and Newton converges quadratically to the (single, physical) solution. This
problem is well behaved. The line search only acts when a full step would *increase* the residual - on
harder problems (several solutions, sharper nonlinearity) that is what separates convergence from
divergence, as the `arctan` example in the notebook shows.

## Exercise 4

**Implement it yourself:** Broyden's method avoids recomputing the Jacobian: after each step update it with $J \leftarrow J + \frac{(\Delta F - J\Delta x)\Delta x^T}{\Delta x^T\Delta x}$. Implement it for the reservoir problem and compare the number of function evaluations with Newton's method.

In [5]:
def broyden(F, x, tol=1e-10, maxiter=100):
    x = np.array(x, float)
    f = np.asarray(F(x)); evals = 1
    J = nonlinear.jacobian(F, x); evals += len(x)          # one finite-difference Jacobian to start
    for it in range(maxiter):
        dx = np.linalg.solve(J, -f)
        x_new = x + dx
        f_new = np.asarray(F(x_new)); evals += 1
        J = J + np.outer(f_new - f - J @ dx, dx) / (dx @ dx)   # rank-one (Broyden) update
        x, f = x_new, f_new
        if np.linalg.norm(f) < tol:
            return x, it + 1, evals
    return x, maxiter, evals

xb, its, ev = broyden(F_const, [90.0, 0.1, 0.0, -0.1])
newton_evals = (sol_const.iterations) * (len(xb) + 1) + 1
print(f"Broyden: {its} iterations, {ev} evaluations of F;  Newton: {sol_const.iterations} iterations, about {newton_evals} evaluations")
print("same solution:", np.allclose(xb, sol_const.x, atol=1e-8))

Broyden: 13 iterations, 18 evaluations of F;  Newton: 6 iterations, about 31 evaluations
same solution: True


Broyden's method needs more iterations - its Jacobian is only an approximation that improves as it goes
(superlinear rather than quadratic convergence) - but each iteration costs a single evaluation of $F$
instead of $n + 1$. When evaluating $F$ is expensive (a simulation, nested root solves) and $n$ is large,
that trade is usually worth it.